# GPT-SoVITS v2 보이스 클로닝 — Colab 학습 (한국어)

**데이터셋:** 육아/훈육 관련 음성에서 추출 → 64 클립 / 약 5.9분 (단일 화자, demucs 잡음제거, large-v3 한국어 전사 완료)

**준비물:** `droh_colab.zip` (이 노트북과 함께 받은 데이터셋 zip, 약 13MB)

> ⚠️ **먼저 GPU 켜기:** 상단 메뉴 → 런타임 → 런타임 유형 변경 → 하드웨어 가속기 **T4 GPU**(이상) 선택 후 저장.
> v2는 T4로 충분합니다. 데이터가 적어 학습은 수십 분 내 끝납니다.

> ⚠️ **데이터가 5.9분으로 적은 편**(권장 10~20분)이라 과적합에 더 취약합니다.
> 에폭을 낮게 유지하고(SoVITS ~8, GPT ~10), 학습 후 결과를 꼭 들어보고 필요하면 에폭을 줄여 재학습하세요.


## 0. GPU 확인


In [ ]:
!nvidia-smi


## 1. 환경 설치 (최초 1회, 약 10~15분)

아래 셀 실행 시 **런타임이 자동 재시작**됩니다. 재시작되면 경고가 떠도 정상이며, **다음 셀부터** 이어서 실행하세요.


In [ ]:
%pip install -q condacolab
import condacolab
condacolab.install_from_url("https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh")
# 실행 후 런타임 자동 재시작 → 아래 셀들 이어서 진행


### 1-2. GPT-SoVITS 클론 + 의존성 + 사전학습 모델 다운로드
`install.sh --source HF` 가 v2 사전학습 가중치까지 자동으로 받습니다.


In [ ]:
%%writefile /content/setup.sh
set -e
cd /content
if [ ! -d GPT-SoVITS ]; then git clone https://github.com/RVC-Boss/GPT-SoVITS.git; fi
cd GPT-SoVITS
if conda env list | awk '{print $1}' | grep -Fxq "GPTSoVITS"; then :; else conda create -n GPTSoVITS python=3.10 -y; fi
source activate GPTSoVITS
pip install ipykernel
bash install.sh --device CU126 --source HF --download-uvr5


In [ ]:
!cd /content && bash setup.sh


## 2. 데이터셋 업로드
실행하면 파일 선택창이 뜹니다. **`droh_colab.zip`** 를 선택하세요.
압축은 `/content/droh/` 로 풀리며, list 안의 경로(`/content/droh/wavs/...`)와 일치합니다.


In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()              # droh_colab.zip 선택
zname = next(iter(up))
with zipfile.ZipFile(zname) as z:
    z.extractall('/content/')
n = len([f for f in os.listdir('/content/droh/wavs') if f.endswith('.wav')])
print(f'추출 완료: wav {n}개')
print('list 첫 줄:', open('/content/droh/droh.colab.list').readline().strip()[:100])


## 3. WebUI 실행 + 학습

아래 셀을 실행하면 출력에 **공개 URL**(`Running on public URL: https://xxxx.gradio.live`)이 뜹니다. 그 링크를 새 탭에서 엽니다.

### WebUI에서 할 일 (순서대로)

**① 버전 선택:** 상단 **Version** 을 `v2` 로 설정 (로컬 추론 가중치와 일치).

**② 1A - 데이터셋 포맷팅** (`1-GPT-SoVITS-TTS` 탭 → `1A` 하위탭)
- `*Text labelling file (텍스트 라벨 파일)` : 
  ```
  /content/droh/droh.colab.list
  ```
- `*Experiment/model name (실험/모델 이름)` : `droh`
- `Audio dataset folder (오디오 폴더)` : **비워둠** (list에 절대경로가 있음)
- **`Start one-click formatting (일괄 포맷팅 시작)`** 클릭 → 1Aa(텍스트)·1Ab(SSL/HuBERT)·1Ac(시맨틱) 자동 진행. 로그에 에러 없이 완료되는지 확인.

**③ 1B - 파인튜닝 학습** (`1B` 하위탭)
- **SoVITS 학습 (s2):** 에폭 기본값(8~15) 두고 `Start SoVITS training` 클릭.
- **GPT 학습 (s1):** 끝나면 `Start GPT training` 클릭. (데이터가 적으니 GPT는 10~15 에폭 권장)
- 두 학습 모두 로그에 `saving ckpt` / 저장 경로가 찍히면 완료.

> 데이터 12분 기준 T4에서 SoVITS + GPT 합쳐 대략 20~40분.
> 과적합 방지: 에폭을 너무 키우지 말 것(SoVITS ~8-12, GPT ~10-15).

> **T4 메모리 주의:** GPT(s1) 학습은 batch 8에서 OOM 날 수 있음 → **batch 4** 권장.
> 가중치 저장 폴더(`SoVITS_weights_v2/`, `logs/droh/logs_s2_v2/`)는 학습 전 **미리 생성**해야 저장 실패가 안 남.


In [ ]:
!cd /content/GPT-SoVITS && source activate GPTSoVITS && export is_share=True && python webui.py


## 4. 학습된 가중치 다운로드
학습 산출물 위치:
- **SoVITS:** `/content/GPT-SoVITS/SoVITS_weights_v2/droh_*.pth`
- **GPT:** `/content/GPT-SoVITS/GPT_weights_v2/droh_*.ckpt`

아래 셀로 내려받은 뒤, 로컬 `voice_clone/infer.py` 의 `t2s_weights_path`(=GPT .ckpt) / `vits_weights_path`(=SoVITS .pth) 를 이 파일들로 교체하면 클로닝 추론에 바로 사용됩니다.

레퍼런스(`REF_AUDIO`/`REF_TEXT`)는 학습 데이터 클립 중 깨끗한 것 하나로 지정하세요 (예: `dataset/droh/wavs/droh_058_6.9s.wav` 와 그 transcript.jsonl 텍스트).


In [ ]:
from google.colab import files
import glob
paths = glob.glob('/content/GPT-SoVITS/SoVITS_weights*/droh*') + glob.glob('/content/GPT-SoVITS/GPT_weights*/droh*')
print('다운로드 대상:', *paths, sep='\n  ')
for p in paths:
    files.download(p)
